# DSA mock data products: DPR-DSA-11 and DPR-DSA-12

This notebook opens the two simulated DSA data products and plots what is in them:

* **DPR-DSA-11** — a channelised **voltage** cutout of a dispersed burst (4+4-bit packed complex samples, two linear polarisations, one start time per channel, coherently dedispersed at a nominal DM).
* **DPR-DSA-12** — a **total-intensity** dynamic spectrum from a formed beam (already incoherently dedispersed, 1 ms sampling).

**Dependencies:** `numpy`, `matplotlib`, `h5py` (needed to read HDF5; nothing else).

**How to run:** set the two file paths in the next cell, then *Kernel → Restart & Run All*. Runtime is a few seconds. Both files are synthetic; nothing here is from a real telescope.

The full schema is defined in the DSA Slack (`#dpr` thread). The files are also self-documenting: every group carries its description in HDF5 *attributes*, and the DSA-11 file has a `README` attribute at the root that explains the layout. The first section prints those so you can read the schema without leaving the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import warnings; warnings.filterwarnings("ignore", message="Mean of empty slice")

from convening2026 import find_product

# All sample products are under data/
F11 = str(find_product("DPR-DSA-11", pattern="**/*DPR-DSA-11*.h5"))
F12 = str(find_product("DPR-DSA-12", pattern="**/*DPR-DSA-12*.h5"))

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False, "font.size": 10})
print("DPR-DSA-11:", F11)
print("DPR-DSA-12:", F12)


## 1. Browse the metadata

`h5py` (and `h5ls`) show the metadata groups as "(0 members)" because they hold only attributes. The helper below prints every attribute of every group and dataset, so the whole schema is visible. The DSA-11 `README` is printed in full: it is the authoritative description of the packing, the per-channel times and what has (and has not) been applied to the data.

In [ ]:
def show_metadata(path, print_readme=True, max_len=90):
    """Print every attribute in the file. Long values are truncated for readability."""
    with h5py.File(path, "r") as f:
        print("=" * 100)
        print(path)
        print("=" * 100)
        if print_readme and "README" in f.attrs:
            print(f.attrs["README"])
            print("-" * 100)
        def fmt(v):
            s = repr(v)
            return s if len(s) <= max_len else s[:max_len] + " ..."
        print("ROOT attributes")
        for k, v in f.attrs.items():
            if k != "README":
                print(f"   {k:32s} = {fmt(v)}")
        def visit(name, obj):
            kind = "dataset" if isinstance(obj, h5py.Dataset) else "group"
            shape = f"  shape={obj.shape} dtype={obj.dtype}" if kind == "dataset" else ""
            print(f"/{name}  [{kind}]{shape}")
            for k, v in obj.attrs.items():
                if k in ("REFERENCE_LIST", "DIMENSION_LIST", "CLASS", "NAME"):
                    continue   # HDF5 dimension-scale plumbing
                print(f"   {k:32s} = {fmt(v)}")
        f.visititems(visit)

show_metadata(F11)

In [ ]:
show_metadata(F12)

## 2. DPR-DSA-11: channelised voltages

Layout `/voltage[chan, pol, sample]` (813 channels × 2 polarisations × 2460 samples, 8.13 µs per sample).
Each byte is one complex sample: high nibble = real, low nibble = imaginary, 4-bit two's complement. Multiplying by `/scale[chan, pol]` restores the relative channel amplitudes (it is *not* a bandpass correction — see the `warning` attribute on `/scale`).

Two things to keep in mind:

1. **Each channel has its own start time** (`/time_unix_int + /time_unix_frac`). The cutout follows the dispersion sweep at the *applied* DM (`/processing.dedispersion_dm`), so sample index `i` is already aligned across channels at that DM.
2. **The best-fit DM (`/burst.dm`) differs slightly from the applied DM.** Only the *difference* is left in the data, as a small residual sweep. Aligning the channels means shifting by the delay of `dm − dedispersion_dm`, not of `dm`.

In [ ]:
def unpack_voltage(path):
    """Return E[chan, pol, sample] as complex64 in the file's 'arbitrary' units, plus axes and masks."""
    with h5py.File(path, "r") as f:
        b = f["voltage"][:]                       # uint8, packed 4+4 bit
        re = ((b >> 4) & 0x0F).astype(np.int8); re[re >= 8] -= 16
        im = ( b       & 0x0F).astype(np.int8); im[im >= 8] -= 16
        E = (re.astype(np.float32) + 1j * im.astype(np.float32)) * f["scale"][:][:, :, None]
        out = dict(
            E=E.astype(np.complex64),
            freq_mhz=f["frequency"][:] / 1e6,
            t0=f["time_unix_int"][:] + f["time_unix_frac"][:],      # per-channel start (UTC s)
            valid=f["valid"][:], flags=f["flags"][:],
            dt=float(f["instrument"].attrs["sample_period_s"]),
            dm_applied=float(f["processing"].attrs["dedispersion_dm"]),
            dm_best=float(f["burst"].attrs["dm"]),
            dm_const=float(f["burst"].attrs["dm_constant"]),
            nu_ref=float(f["burst"].attrs["dm_reference_frequency_mhz"]),
            offpulse=f["scale"].attrs["offpulse_samples"],
            pols=[p.decode() if isinstance(p, bytes) else str(p) for p in f["polarization"][:]],
        )
    return out

d11 = unpack_voltage(F11)
E = d11["E"]
print("E shape (chan, pol, sample):", E.shape, "| pols:", d11["pols"])
print(f"band {d11['freq_mhz'][0]:.1f}-{d11['freq_mhz'][-1]:.1f} MHz, {E.shape[0]} channels, "
      f"{E.shape[2]*d11['dt']*1e3:.1f} ms per channel window")
print(f"applied DM {d11['dm_applied']}, best-fit DM {d11['dm_best']}  ->  residual {d11['dm_best']-d11['dm_applied']:+.3f} pc cm^-3")
print(f"flagged channels: {(d11['flags'] != 0).sum()}  (bit0=RFI: {(d11['flags'] & 1 > 0).sum()}, bit1=band edge: {(d11['flags'] & 2 > 0).sum()})")
print(f"invalid samples: {(~d11['valid']).sum()}")

### 2.1 Quicklook (stored in the file, display only)

`/quicklook/stokes_i` is a decimated, per-channel S/N-normalised Stokes I, already at the best-fit DM. It is the fastest way to see the burst; it must not be used for measurements.

In [ ]:
with h5py.File(F11, "r") as f:
    ql = f["quicklook/stokes_i"][:]
    ql_t = f["quicklook/time_offset_s"][:] * 1e3
    ql_dm = f["quicklook/stokes_i"].attrs["dedispersed_at_dm"]
ql = ql.copy(); ql[d11["flags"] != 0] = np.nan           # blank flagged channels (not done in the file)

fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True, gridspec_kw=dict(height_ratios=[1, 2.5]))
ax[0].plot(ql_t, np.nanmean(ql, axis=0), color="k")
ax[0].set_ylabel("mean S/N per channel")
ax[0].set_title(f"DPR-DSA-11 quicklook (dedispersed at DM = {ql_dm})")
im = ax[1].imshow(ql, aspect="auto", origin="lower",
                  extent=[ql_t[0], ql_t[-1], d11["freq_mhz"][0], d11["freq_mhz"][-1]],
                  vmin=-1, vmax=np.nanpercentile(ql, 99.5), cmap="viridis")
ax[1].set_xlabel("time relative to burst peak at reference frequency (ms)")
ax[1].set_ylabel("frequency (MHz)")
plt.colorbar(im, ax=ax[1], label="S/N (sigma)")
plt.tight_layout(); plt.show()

### 2.2 Stokes I from the voltages, aligned at the best-fit DM

Steps: form Stokes I = |E_X|² + |E_Y|², blank invalid samples, shift each channel by the **residual** dispersion delay (fill with NaN rather than wrapping), then normalise each channel by its off-pulse statistics for display. The off-pulse windows are the ones recorded on `/scale` (`offpulse_samples`).

In [ ]:
def residual_shift_samples(d):
    """Samples to shift each channel so that the burst aligns at the best-fit DM."""
    ddm = d["dm_best"] - d["dm_applied"]
    delay_s = d["dm_const"] * ddm * (d["freq_mhz"]**-2 - d["nu_ref"]**-2)   # >0 at lower frequencies
    return np.round(delay_s / d["dt"]).astype(int)

def shift_nan(x, n):
    """Shift a 1-D array left by n samples (n>0 -> earlier), filling with NaN. No wrap-around."""
    y = np.full_like(x, np.nan)
    if n >= 0:
        y[:len(x)-n] = x[n:]
    else:
        y[-n:] = x[:len(x)+n]
    return y

I = (np.abs(E[:, 0, :])**2 + np.abs(E[:, 1, :])**2).astype(np.float32)
I[~d11["valid"]] = np.nan
shift = residual_shift_samples(d11)
I_al = np.vstack([shift_nan(I[c], shift[c]) for c in range(I.shape[0])])
print(f"residual shift ranges from {shift.min()} to {shift.max()} samples "
      f"({shift.max()*d11['dt']*1e6:.0f} us at the bottom of the band)")

# per-channel off-pulse normalisation for display
a, b, c, e = d11["offpulse"]
off = np.concatenate([I_al[:, a:b], I_al[:, c:e]], axis=1)
mu, sd = np.nanmean(off, axis=1, keepdims=True), np.nanstd(off, axis=1, keepdims=True)
I_disp = (I_al - mu) / sd
I_disp[d11["flags"] != 0] = np.nan

t_ms = np.arange(I.shape[1]) * d11["dt"] * 1e3
prof = np.nanmean(I_disp, axis=0)
peak = np.nanargmax(prof)
spec = np.nanmean(I_disp[:, peak-8:peak+8], axis=1)   # spectrum around the peak (+-65 us)

fig = plt.figure(figsize=(9, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1, 2.5], hspace=0.05, wspace=0.05)
axp = fig.add_subplot(gs[0, 0]); axw = fig.add_subplot(gs[1, 0], sharex=axp); axs = fig.add_subplot(gs[1, 1], sharey=axw)
axp.plot(t_ms - t_ms[peak], prof, color="k"); axp.set_ylabel("mean S/N"); axp.tick_params(labelbottom=False)
axp.set_title(f"DPR-DSA-11 Stokes I from /voltage, aligned at DM = {d11['dm_best']}")
imw = axw.imshow(I_disp, aspect="auto", origin="lower",
                 extent=[t_ms[0]-t_ms[peak], t_ms[-1]-t_ms[peak], d11["freq_mhz"][0], d11["freq_mhz"][-1]],
                 vmin=-1, vmax=np.nanpercentile(I_disp, 99.7), cmap="viridis")
axw.set_xlabel("time relative to peak (ms)"); axw.set_ylabel("frequency (MHz)")
axw.set_xlim(-3, 3)
axs.plot(spec, d11["freq_mhz"], color="k", lw=0.8); axs.set_xlabel("S/N at peak"); axs.tick_params(labelleft=False)
plt.show()

### 2.3 Polarisation check (uncalibrated)

With two linear feeds, Q = |X|² − |Y|², U = 2 Re(X Y\*), V = 2 Im(X Y\*) up to sign conventions. Nothing is polarisation-calibrated (`/calibration` says so), so this is a qualitative look only. Averaged over the burst, the fractional Q and U should oscillate with λ² if the burst is Faraday rotated (`/burst.rotation_measure_rad_m2`).

In [ ]:
X, Y = E[:, 0, :], E[:, 1, :]
on = slice(peak - 120, peak + 120)                    # +-1 ms around the peak (aligned sample indices)
# Apply the same residual shift to the complex voltages so the on-pulse window is common to all channels
Xa = np.vstack([shift_nan(X[c], shift[c]) for c in range(X.shape[0])])
Ya = np.vstack([shift_nan(Y[c], shift[c]) for c in range(Y.shape[0])])
XX = np.nanmean(np.abs(Xa[:, on])**2, axis=1); YY = np.nanmean(np.abs(Ya[:, on])**2, axis=1)
XY = np.nanmean(Xa[:, on] * np.conj(Ya[:, on]), axis=1)
# subtract the off-pulse (noise) power so I, Q are the burst's own
XXo = np.nanmean(np.abs(Xa[:, a:b])**2, axis=1); YYo = np.nanmean(np.abs(Ya[:, a:b])**2, axis=1)
Ib = (XX - XXo) + (YY - YYo); Qb = (XX - XXo) - (YY - YYo); Ub = 2 * XY.real; Vb = 2 * XY.imag
good = (d11["flags"] == 0) & (Ib > 0)

lam2 = (299.792458 / d11["freq_mhz"])**2               # wavelength^2 in m^2

# single channels are too noisy: average I, Q, U in blocks of NB channels (flagged channels excluded)
NB = 24
def block(x, w):
    n = len(x) // NB
    xs = np.where(w, x, 0.0)[:n*NB].reshape(n, NB); ws = w[:n*NB].reshape(n, NB)
    return xs.sum(1) / np.maximum(ws.sum(1), 1)
Ibk, Qbk, Ubk, l2k = (block(v, good) for v in (Ib, Qb, Ub, lam2))

fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.plot(l2k, Qbk / Ibk, "o-", ms=4, label="Q/I")
ax.plot(l2k, Ubk / Ibk, "s-", ms=4, label="U/I")
ax.set_xlabel(r"$\lambda^2$ (m$^2$)"); ax.set_ylabel("fractional polarisation (uncalibrated)")
ax.set_ylim(-0.6, 0.6); ax.legend(); ax.set_title(f"DPR-DSA-11: Q/I and U/I, averaged over the burst and {NB}-channel blocks")
plt.tight_layout(); plt.show()
with h5py.File(F11, "r") as f:
    print("file says RM =", f["burst"].attrs["rotation_measure_rad_m2"], "rad m^-2,",
          "linear fraction =", f["burst"].attrs["linear_polarization_fraction"])

## 3. DPR-DSA-12: total-intensity dynamic spectrum

`/dynamic_spectrum[chan, time]` is a detected, incoherently dedispersed beam (813 channels × 64 samples at 1 ms). One time per sample is stored (`/time_unix_int + /time_unix_frac`), common to all channels — unlike DSA-11 there is no per-channel window. `/valid` marks samples not received; `/flags` marks suspect channels.

In [ ]:
with h5py.File(F12, "r") as f:
    D = f["dynamic_spectrum"][:].astype(np.float32)
    fr12 = f["frequency"][:] / 1e6
    t12 = f["time_unix_int"][:] + f["time_unix_frac"][:]
    valid12 = f["valid"][:]; flags12 = f["flags"][:]
    ql12 = f["quicklook/stokes_i"][:]
    b12 = dict(f["burst"].attrs); p12 = dict(f["processing"].attrs); pt12 = dict(f["pointing"].attrs)

print(f"{D.shape[0]} channels x {D.shape[1]} samples, dt = {np.median(np.diff(t12))*1e3:.3f} ms, "
      f"band {fr12[0]:.1f}-{fr12[-1]:.1f} MHz")
print(f"dedispersed at DM = {p12['dedispersion_dm']} (ref {p12['dedispersion_reference_frequency_mhz']} MHz); "
      f"burst DM {b12['dm']} +- {b12['dm_error']}, S/N {b12['snr']}, width {b12['width_fwhm_s']*1e3:.2f} ms")
print(f"pointing RA, Dec = {pt12['ra_deg']:.4f}, {pt12['dec_deg']:.4f} ({pt12['frame']})")
print(f"invalid samples: {(~valid12).sum()} ; flagged channels: {(flags12 != 0).sum()} ; NaNs in data: {np.isnan(D).sum()}")

Dm = D.copy(); Dm[~valid12] = np.nan; Dm[flags12 != 0] = np.nan
t_rel = (t12 - t12[0]) * 1e3
prof12 = np.nanmean(Dm, axis=0); pk = np.nanargmax(prof12)
spec12 = np.nanmean(Dm[:, max(pk-1, 0):pk+2], axis=1)

fig = plt.figure(figsize=(9, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1, 2.5], hspace=0.05, wspace=0.05)
axp = fig.add_subplot(gs[0, 0]); axw = fig.add_subplot(gs[1, 0], sharex=axp); axs = fig.add_subplot(gs[1, 1], sharey=axw)
axp.plot(t_rel, prof12, color="k", drawstyle="steps-mid"); axp.set_ylabel("mean intensity"); axp.tick_params(labelbottom=False)
axp.set_title(f"DPR-DSA-12 dynamic spectrum (dedispersed at DM = {p12['dedispersion_dm']})")
imw = axw.imshow(Dm, aspect="auto", origin="lower", extent=[t_rel[0], t_rel[-1], fr12[0], fr12[-1]],
                 vmin=np.nanpercentile(Dm, 1), vmax=np.nanpercentile(Dm, 99.5), cmap="viridis")
axw.set_xlabel("time since first sample (ms)"); axw.set_ylabel("frequency (MHz)")
axs.plot(spec12, fr12, color="k", lw=0.8); axs.set_xlabel("intensity at peak"); axs.tick_params(labelleft=False)
plt.show()

# the stored quicklook, for comparison
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.imshow(ql12, aspect="auto", origin="lower", extent=[0, ql12.shape[1], fr12[0], fr12[-1]], cmap="viridis")
ax.set_xlabel("decimated time bin"); ax.set_ylabel("frequency (MHz)"); ax.set_title("DPR-DSA-12 /quicklook/stokes_i")
plt.tight_layout(); plt.show()

## 4. Side-by-side summary

| | DPR-DSA-11 | DPR-DSA-12 |
|---|---|---|
| Content | complex voltages, 2 linear pols | detected total intensity |
| Shape | (813 chan, 2 pol, 2460 samp) | (813 chan, 64 samp) |
| Time resolution | 8.13 µs | 1 ms |
| Time axis | one start time **per channel** (dispersion-sweep cutout) | one time per sample, common to all channels |
| Dedispersion | coherent at the applied DM; residual sweep left in | incoherent at the best-fit DM |
| Display-only product | `/quicklook/stokes_i` | `/quicklook/stokes_i` |
| Missing vs suspect | `/valid` (missing) vs `/flags` (bitmask, suspect) | same |

**Things that trip people up** (all stated in the files' attributes): use the *difference* between best-fit and applied DM when aligning DSA-11; shift with NaN fill, not `np.roll`; `/scale` restores relative amplitudes and must not be divided out again; `/flags` is a bitmask — test bits, don't compare values; the metadata groups are not empty, they hold attributes.